<a href="https://colab.research.google.com/github/luca-fce/siniestros-futbol-caba/blob/main/notebook/3_1_seleccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partidos de la Selección Argentina (2021–2025)

La base de resultados internacionales ([Kaggle](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)) contiene el historial global de partidos entre 1872 y 2026. Para acotar el alcance al del trabajo práctico aplicamos dos filtros: **temporal (2021–2025)** y **de variable (partidos de la Selección Argentina)**. Incluimos partidos oficiales y amistosos.

> *Uso de IA:* notebook armado con asistencia de Claude (Anthropic). Prompt: «Filtrá los partidos de la Selección 2021–2025 de la base de resultados internacionales, agregá la hora real de inicio en hora argentina y el resultado final con penales, y exportalo en el formato del cruce con siniestros». El grupo revisó los horarios contra la fuente y validó las interpretaciones.

## 1. Carga y exploración inicial

In [1]:
import pandas as pd
import numpy as np

BASE = 'https://raw.githubusercontent.com/luca-fce/siniestros-futbol-caba/main/datos/Resultados%20Internacionales%201872%20-%202026/'
df_futbol = pd.read_csv(BASE + 'results.csv', parse_dates=['date'])
penales_raw = pd.read_csv(BASE + 'shootouts.csv', parse_dates=['date'])
print('Base completa:', df_futbol.shape)
df_futbol.tail()

Base completa: (49945, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49940,2026-10-06,Belize,French Guiana,0,1,CONCACAF Nations League,Belmopan,Belize,False
49941,2026-10-06,Turks and Caicos Islands,Montserrat,1,4,CONCACAF Nations League,Providenciales,Turks and Caicos Islands,False
49942,2026-10-06,Antigua and Barbuda,Aruba,3,2,CONCACAF Nations League,North Sound,Antigua and Barbuda,False
49943,2026-10-06,Saint Martin,Bahamas,6,0,CONCACAF Nations League,St Croix,United States Virgin Islands,True
49944,2026-10-06,Saudi Arabia,United Arab Emirates,2,0,Gulf Cup,Jeddah,Saudi Arabia,False


In [2]:
filtro_fechas = df_futbol['date'].between('2021-01-01', '2025-12-31')
filtro_argentina = (df_futbol['home_team'] == 'Argentina') | (df_futbol['away_team'] == 'Argentina')
arg = df_futbol[filtro_fechas & filtro_argentina].copy().reset_index(drop=True)

print(f'Partidos de Argentina 2021-2025: {len(arg)}')
print('\nTipos de dato:'); print(arg.dtypes)
print('\nNulos:'); print(arg.isnull().sum())
print('\nDuplicados:', arg.duplicated().sum())
arg.head()

Partidos de Argentina 2021-2025: 67

Tipos de dato:
date          datetime64[us]
home_team                str
away_team                str
home_score             int64
away_score             int64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

Nulos:
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

Duplicados: 0


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2021-06-03,Argentina,Chile,1,1,FIFA World Cup qualification,Santiago del Estero,Argentina,False
1,2021-06-08,Colombia,Argentina,2,2,FIFA World Cup qualification,Barranquilla,Colombia,False
2,2021-06-14,Argentina,Chile,1,1,Copa América,Rio de Janeiro,Brazil,True
3,2021-06-18,Argentina,Uruguay,1,0,Copa América,Brasília,Brazil,True
4,2021-06-21,Argentina,Paraguay,1,0,Copa América,Brasília,Brazil,True


In [3]:
arg['tournament'].value_counts()

tournament
FIFA World Cup qualification      31
Friendly                          15
Copa América                      13
FIFA World Cup                     7
CONMEBOL–UEFA Cup of Champions     1
Name: count, dtype: int64

**Hallazgos:** 67 partidos sin nulos ni duplicados (31 de Eliminatorias, 15 amistosos, 13 de Copa América, 7 del Mundial y la Finalissima). El partido suspendido contra Brasil en São Paulo (5/9/2021) no figura, lo cual es correcto porque no tuvo resultado.

El problema principal es que **la base no tiene la hora de los partidos**, y sin hora no se puede armar la ventana de 12 h. Además, los goles están como local/visitante y no desde el punto de vista de Argentina, y los partidos definidos por penales figuran como empate.

## 2. Horario de cada partido

Tomamos la hora de inicio de cada partido de [Wikipedia: Argentina national football team results (2020–present)](https://en.wikipedia.org/wiki/Argentina_national_football_team_results_(2020%E2%80%93present)), que informa la **hora local del estadio con su huso horario** (UTC). La pasamos a **hora argentina (UTC−3**, Argentina no tiene horario de verano): `hora_arg = hora_local − (huso + 3)`.

Ejemplo: el debut con Arabia Saudita en Qatar fue a las 13:00 (UTC+3) → 13 − 6 = **07:00** en Argentina.

*Corrección manual:* para Argentina–Chile del 5/9/2024 en el Monumental la fuente indica UTC−4, pero Buenos Aires es UTC−3; se usó 21:00 hora argentina.

In [4]:
# fecha: (hora local de inicio, huso horario UTC del estadio)
horarios = {
    '2021-06-03': ('21:00', -3), '2021-06-08': ('18:00', -5), '2021-06-14': ('18:00', -3),
    '2021-06-18': ('21:00', -3), '2021-06-21': ('21:00', -3), '2021-06-28': ('20:00', -4),
    '2021-07-03': ('22:00', -3), '2021-07-06': ('22:00', -3), '2021-07-10': ('21:00', -3),
    '2021-09-02': ('20:00', -4), '2021-09-09': ('20:30', -3), '2021-10-07': ('20:30', -3),
    '2021-10-10': ('20:30', -3), '2021-10-14': ('20:30', -3), '2021-11-12': ('20:30', -3),
    '2021-11-16': ('20:30', -3), '2022-01-27': ('21:15', -3), '2022-02-01': ('20:30', -3),
    '2022-03-25': ('20:30', -3), '2022-03-29': ('18:30', -5), '2022-06-01': ('19:45', 1),
    '2022-06-05': ('19:00', 2), '2022-09-23': ('21:00', -4), '2022-09-27': ('21:00', -4),
    '2022-11-16': ('19:30', 4), '2022-11-22': ('13:00', 3), '2022-11-26': ('22:00', 3),
    '2022-11-30': ('22:00', 3), '2022-12-03': ('22:00', 3), '2022-12-09': ('22:00', 3),
    '2022-12-13': ('22:00', 3), '2022-12-18': ('18:00', 3), '2023-03-23': ('20:30', -3),
    '2023-03-28': ('20:30', -3), '2023-06-15': ('20:00', 8), '2023-06-19': ('19:30', 7),
    '2023-09-07': ('21:00', -3), '2023-09-12': ('16:00', -4), '2023-10-12': ('21:00', -3),
    '2023-10-17': ('21:00', -5), '2023-11-16': ('21:00', -3), '2023-11-21': ('21:30', -3),
    '2024-03-22': ('20:00', -4), '2024-03-26': ('19:50', -7), '2024-06-09': ('17:00', -5),
    '2024-06-14': ('20:00', -4), '2024-06-20': ('20:00', -4), '2024-06-25': ('21:00', -4),
    '2024-06-29': ('20:00', -4), '2024-07-04': ('20:00', -5), '2024-07-09': ('20:00', -4),
    '2024-07-14': ('21:22', -4), '2024-09-05': ('21:00', -3), '2024-09-10': ('17:30', -5),
    '2024-10-10': ('17:00', -4), '2024-10-15': ('21:00', -3), '2024-11-14': ('20:30', -3),
    '2024-11-19': ('21:45', -3), '2025-03-21': ('20:30', -3), '2025-03-25': ('21:00', -3),
    '2025-06-05': ('21:00', -4), '2025-06-10': ('21:00', -3), '2025-09-04': ('20:30', -3),
    '2025-09-09': ('18:00', -5), '2025-10-10': ('20:00', -4), '2025-10-14': ('18:00', -4),
    '2025-11-14': ('17:00', 1),
}
# Partidos con alargue (90' empatados y se jugó tiempo extra)
con_alargue = {'2022-12-09', '2022-12-18', '2024-07-14'}

arg['fecha'] = arg['date'].dt.strftime('%Y-%m-%d')
faltan = set(arg['fecha']) - set(horarios)
print('Partidos sin horario:', len(faltan), faltan or '')

arg['hora_local'] = arg['fecha'].map(lambda f: horarios[f][0])
arg['huso_utc'] = arg['fecha'].map(lambda f: horarios[f][1])
inicio_local = pd.to_datetime(arg['fecha'] + ' ' + arg['hora_local'])
arg['inicio'] = inicio_local - pd.to_timedelta(arg['huso_utc'] + 3, unit='h')   # a hora argentina
arg['hora_arg'] = arg['inicio'].dt.strftime('%H:%M')
arg[['fecha', 'home_team', 'away_team', 'city', 'hora_local', 'huso_utc', 'inicio']].head(10)

Partidos sin horario: 0 


,fecha,home_team,away_team,city,hora_local,huso_utc,inicio
0,2021-06-03,Argentina,Chile,Santiago del Estero,21:00,-3,2021-06-03 21:00:00
1,2021-06-08,Colombia,Argentina,Barranquilla,18:00,-5,2021-06-08 20:00:00
2,2021-06-14,Argentina,Chile,Rio de Janeiro,18:00,-3,2021-06-14 18:00:00
3,2021-06-18,Argentina,Uruguay,Brasília,21:00,-3,2021-06-18 21:00:00
4,2021-06-21,Argentina,Paraguay,Brasília,21:00,-3,2021-06-21 21:00:00
5,2021-06-28,Argentina,Bolivia,Cuiabá,20:00,-4,2021-06-28 21:00:00
6,2021-07-03,Argentina,Ecuador,Goiânia,22:00,-3,2021-07-03 22:00:00
7,2021-07-06,Argentina,Colombia,Brasília,22:00,-3,2021-07-06 22:00:00
8,2021-07-10,Brazil,Argentina,Rio de Janeiro,21:00,-3,2021-07-10 21:00:00
9,2021-09-02,Venezuela,Argentina,Caracas,20:00,-4,2021-09-02 21:00:00


## 3. Resultado desde el punto de vista de Argentina

1. **Goles a favor y en contra** según si Argentina fue local o visitante.
2. **Condición:** local (en Argentina), visitante o neutral.
3. **Penales:** usamos `shootouts.csv` (misma fuente) para saber quién ganó la tanda. Para el hincha, ganar por penales es una victoria y perder por penales, una derrota.

In [5]:
es_local = arg['home_team'] == 'Argentina'
arg['rival'] = np.where(es_local, arg['away_team'], arg['home_team'])
arg['goles_a_favor'] = np.where(es_local, arg['home_score'], arg['away_score'])
arg['goles_en_contra'] = np.where(es_local, arg['away_score'], arg['home_score'])
arg['condicion'] = np.select([arg['neutral'], es_local], ['Neutral', 'Home'], 'Away')

pen = penales_raw[(penales_raw['home_team'] == 'Argentina') | (penales_raw['away_team'] == 'Argentina')]
ganador_penales = pen.set_index(pen['date'].dt.strftime('%Y-%m-%d'))['winner']
arg['penales'] = arg['fecha'].isin(ganador_penales.index)
arg['alargue'] = arg['fecha'].isin(con_alargue)

arg['resultado'] = np.select([arg['goles_a_favor'] > arg['goles_en_contra'],
                              arg['goles_a_favor'] < arg['goles_en_contra']], ['Victoria', 'Derrota'], 'Empate')
gano = arg['fecha'].map(ganador_penales) == 'Argentina'
arg.loc[arg['penales'] & gano, 'resultado'] = 'Victoria'
arg.loc[arg['penales'] & ~gano, 'resultado'] = 'Derrota'

arg[arg['penales'] | arg['alargue']][['fecha', 'rival', 'tournament', 'goles_a_favor', 'goles_en_contra',
                                      'alargue', 'penales', 'resultado']]

,fecha,rival,tournament,goles_a_favor,goles_en_contra,alargue,penales,resultado
7,2021-07-06,Colombia,Copa América,1,1,False,True,Victoria
29,2022-12-09,Netherlands,FIFA World Cup,2,2,True,True,Victoria
31,2022-12-18,France,FIFA World Cup,3,3,True,True,Victoria
49,2024-07-04,Ecuador,Copa América,1,1,False,True,Victoria
51,2024-07-14,Colombia,Copa América,1,0,True,False,Victoria


## 4. Resumen agrupado

In [6]:
seleccion = (arg.rename(columns={'tournament': 'competencia'})
             [['fecha', 'hora_arg', 'inicio', 'rival', 'competencia', 'condicion', 'city', 'country',
               'goles_a_favor', 'goles_en_contra', 'resultado', 'alargue', 'penales']]
             .sort_values('inicio').reset_index(drop=True))

print(seleccion['resultado'].value_counts(), '\n')
pd.crosstab(seleccion['competencia'], seleccion['resultado'], margins=True)

resultado
Victoria    54
Empate       8
Derrota      5
Name: count, dtype: int64 



resultado,Derrota,Empate,Victoria,All
competencia,,,,
CONMEBOL–UEFA Cup of Champions,0,0,1,1
Copa América,0,1,12,13
FIFA World Cup,1,0,6,7
FIFA World Cup qualification,4,7,20,31
Friendly,0,0,15,15
All,5,8,54,67


In [7]:
# Las 5 derrotas
seleccion[seleccion['resultado'] == 'Derrota'][['fecha', 'hora_arg', 'rival', 'competencia', 'condicion', 'goles_a_favor', 'goles_en_contra']]

,fecha,hora_arg,rival,competencia,condicion,goles_a_favor,goles_en_contra
25,2022-11-22,07:00,Saudi Arabia,FIFA World Cup,Neutral,1,2
40,2023-11-16,21:00,Uruguay,FIFA World Cup qualification,Home,0,2
53,2024-09-10,19:30,Colombia,FIFA World Cup qualification,Away,1,2
56,2024-11-14,20:30,Paraguay,FIFA World Cup qualification,Away,1,2
63,2025-09-09,20:00,Ecuador,FIFA World Cup qualification,Away,0,1


In [8]:
# ¿A qué hora (argentina) empezaron los partidos?
seleccion['inicio'].dt.hour.value_counts().sort_index()

inicio
7      1
9      2
12     2
13     1
14     1
15     1
16     5
17     1
18     2
19     3
20    16
21    22
22     8
23     2
Name: count, dtype: int64

**Interpretación.** Entre 2021 y 2025 la Selección ganó 54 de 67 partidos (4 de ellos por penales), empató 8 y perdió solo **5** (Arabia Saudita en el Mundial y cuatro de Eliminatorias). Ganó todas las tandas de penales que jugó, así que no agregan derrotas. Con tan pocas derrotas, el efecto de la Selección no puede evaluarse estadísticamente por separado: en el análisis se suma a River y Boca y sus 5 derrotas se describen caso por caso.

La mayoría de los partidos empezó entre las 20 y las 22 h (hora argentina), pero hay excepciones importantes: los partidos del Mundial de Qatar (07:00, 12:00 y 16:00 h), la Finalissima en Londres (15:45 h) y amistosos en Asia (09:00 y 09:30 h en Beijing y Jakarta). Asignar una hora fija a todos habría ubicado mal la ventana de 12 h en esos casos.

## 5. Exportación

Guardamos la tabla con el formato que usa el notebook de análisis (`3_2_cruce_partidos_siniestros.ipynb`).

In [9]:
seleccion.drop(columns=['inicio']).to_csv('partidos_seleccion_2021_2025.csv', index=False)
print('Guardado partidos_seleccion_2021_2025.csv:', seleccion.shape[0], 'partidos')

Guardado partidos_seleccion_2021_2025.csv: 67 partidos
